# Prep 3 · Bayesian inference by hand

**Time:** about 90 minutes. **Needs:** NumPy and matplotlib only — no JAX, no NumPyro.

Bayes' rule in one line:

$$p(\text{unknown} \mid \text{data}) \;\propto\; p(\text{data} \mid \text{unknown}) \; p(\text{unknown})$$

**posterior ∝ likelihood × prior.** The prior says what you believed before; the likelihood says how well
each candidate value explains the data; the posterior is what you believe afterwards. Everything in this
notebook is computed on a **grid** — brute force, no libraries — so there is nothing to trust but
arithmetic. By the end you will have solved the MRI reconstruction problem in two pixels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. A coin

We flip a coin 10 times and see 7 heads. What is the probability of heads, `p`?

- **Unknown:** `p ∈ [0, 1]`. We lay a grid of 1001 candidate values.
- **Prior:** we know nothing, so every `p` is equally likely (uniform).
- **Likelihood:** the probability of the data for a given `p`: `p^7 (1 − p)^3` (up to a constant).
- **Posterior:** multiply, then normalise so it sums to 1.

In [ ]:
p = np.linspace(0, 1, 1001)
prior = np.ones_like(p)                     # uniform
likelihood = p ** 7 * (1 - p) ** 3          # 7 heads, 3 tails

### Exercise 1 — the posterior on a grid

Write `grid_posterior(prior, likelihood)` returning the normalised posterior (it must sum to 1), then
compute the **MAP** (the grid value where the posterior is highest) and the **posterior mean**
(`Σ p · posterior`). They differ — think about why.

In [ ]:
def grid_posterior(prior, likelihood):
    # YOUR CODE HERE: multiply, then normalise so the result sums to 1
    ...

posterior = grid_posterior(prior, likelihood)
p_map = ...      # YOUR CODE HERE: the grid value where `posterior` is largest
p_mean = ...     # YOUR CODE HERE: sum of p * posterior

In [ ]:
assert posterior is not ... and p_map is not ... and p_mean is not ..., "fill in the exercise above first"
print(f"MAP = {p_map:.3f}   posterior mean = {p_mean:.3f}")
plt.plot(p, posterior); plt.axvline(p_map, ls="--", label=f"MAP {p_map:.2f}"); plt.axvline(p_mean, ls=":", label=f"mean {p_mean:.2f}")
plt.xlabel("p"); plt.ylabel("posterior"); plt.legend(); plt.show()

# check (the exact answers: MAP = 7/10, mean = 8/12)
assert abs(posterior.sum() - 1) < 1e-9
assert abs(p_map - 0.7) < 0.005 and abs(p_mean - 2 / 3) < 0.005
print("exercise 1 OK")

### The prior matters when data are scarce — and stops mattering when they are not

Same coin, three priors: uniform, mildly-fair (`p(1−p)`), strongly-fair (`p^20 (1−p)^20`). With 10 flips the
posteriors disagree; with 1000 flips (700 heads) they agree. **This is the whole game in MRI:** with
little data (an undersampled scan) the prior decides; our job is to learn a prior that is *right*.

In [ ]:
assert grid_posterior(prior, likelihood) is not None, "finish exercise 1 first"
priors = {"uniform": np.ones_like(p), "mildly fair": p * (1 - p), "strongly fair": p ** 20 * (1 - p) ** 20}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, (h, n_flips) in zip(ax, ((7, 10), (700, 1000))):
    like = np.exp(h * np.log(p + 1e-12) + (n_flips - h) * np.log(1 - p + 1e-12))
    for name, pr in priors.items():
        a.plot(p, grid_posterior(pr, like), label=name)
    a.set_title(f"{h} heads in {n_flips} flips"); a.set_xlim(0.3, 1); a.legend()
plt.tight_layout(); plt.show()

## 2. Denoising one number — the prior *shrinks*

Now a continuous unknown: a true value `x` with prior `x ~ N(0, τ²)`, and a noisy measurement
`y = x + ε`, `ε ~ N(0, σ²)`. This has a closed form — the posterior is Gaussian with

$$\text{mean} = \frac{\tau^2}{\tau^2 + \sigma^2}\, y, \qquad \text{std} = \Big(\frac{1}{\tau^2} + \frac{1}{\sigma^2}\Big)^{-1/2}$$

so the estimate is the measurement **shrunk toward the prior mean**, by an amount set by how noisy the
data are relative to how confident the prior is.

### Exercise 2 — check the formula on a grid

With `τ = 1, σ = 0.5, y = 1.2`, compute the posterior on the grid `xs` (Gaussian prior × Gaussian
likelihood), then its mean and standard deviation. Compare to the formula.

In [ ]:
tau, sigma, y = 1.0, 0.5, 1.2
xs = np.linspace(-4, 4, 4001)

prior_x = ...     # YOUR CODE HERE: Gaussian prior density on xs (unnormalised is fine)
like_x = ...      # YOUR CODE HERE: Gaussian likelihood of y for each xs
post_x = grid_posterior(prior_x, like_x)

post_mean = ...   # YOUR CODE HERE
post_std = ...    # YOUR CODE HERE  (sqrt of the posterior variance)

In [ ]:
assert post_mean is not ... and post_std is not ..., "fill in the exercise above first"
formula_mean = tau ** 2 / (tau ** 2 + sigma ** 2) * y
formula_std = (1 / tau ** 2 + 1 / sigma ** 2) ** -0.5
print(f"grid: mean {post_mean:.3f}, std {post_std:.3f}   formula: mean {formula_mean:.3f}, std {formula_std:.3f}")
plt.plot(xs, prior_x / prior_x.sum(), label="prior"); plt.plot(xs, like_x / like_x.sum(), label="likelihood"); plt.plot(xs, post_x, label="posterior")
plt.axvline(y, color="k", ls=":", label="measurement y"); plt.xlim(-2.5, 3); plt.legend(); plt.show()

assert abs(post_mean - formula_mean) < 0.01 and abs(post_std - formula_std) < 0.01
print("exercise 2 OK")

## 3. The whole project in two pixels

An "image" with two pixels, `x = (x₁, x₂)`. The "scanner" measures only their **sum**:
`y = x₁ + x₂ + ε`, `ε ~ N(0, σ²)`, and we observe `y = 1.0`.

One number measured, two unknown: **ill-posed**. The likelihood alone can't tell `(0.2, 0.8)` from
`(0.8, 0.2)` from `(5, −4)` — it is a *ridge* in the `(x₁, x₂)` plane. A prior turns the ridge into a bump:

- **Prior A (independent):** `x₁, x₂ ~ N(0, 1)` each.
- **Prior B (smooth):** the same, plus `x₁ − x₂ ~ N(0, 0.1²)` — "neighbouring pixels are similar".

This is exactly undersampled MRI: the measurement pins down some directions of image space, the prior
has to supply the rest, and the posterior *width* tells you which directions you should not trust.

In [ ]:
sigma_y, y_obs = 0.1, 1.0
g = np.linspace(-4, 4, 801)          # wide enough that the prior is not cut off
X1, X2 = np.meshgrid(g, g, indexing="ij")

likelihood_2d = np.exp(-0.5 * (y_obs - (X1 + X2)) ** 2 / sigma_y ** 2)     # a ridge
prior_A = np.exp(-0.5 * (X1 ** 2 + X2 ** 2))
prior_B = prior_A * np.exp(-0.5 * (X1 - X2) ** 2 / 0.1 ** 2)

plt.figure(figsize=(4, 4)); plt.contourf(g, g, likelihood_2d.T, 20); plt.xlim(-2, 2); plt.ylim(-2, 2); plt.title("likelihood alone: a ridge"); plt.xlabel("x1"); plt.ylabel("x2"); plt.show()

### Exercise 3 — posterior, MAP, and per-pixel uncertainty under each prior

For each prior compute the normalised 2-D posterior, its MAP `(x₁, x₂)`, and the **marginal** mean and
standard deviation of each pixel (sum the posterior over the *other* axis first). Fill in `summarise`.
Then look at the numbers: which prior leaves the pixels more uncertain, and why?

In [ ]:
def summarise(prior):
    post = prior * likelihood_2d
    post = ...            # YOUR CODE HERE: normalise
    assert post is not ..., "normalise `post` first"
    i, j = np.unravel_index(np.argmax(post), post.shape)
    x_map = np.array([g[i], g[j]])
    m1, m2 = ..., ...     # YOUR CODE HERE: marginals (sum over axis 1 for x1, over axis 0 for x2)
    mean = ...            # YOUR CODE HERE: array of the two marginal means
    std = ...             # YOUR CODE HERE: array of the two marginal standard deviations
    return post, x_map, mean, std

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
results = {}
for a, (name, pr) in zip(ax, (("A: independent", prior_A), ("B: smooth", prior_B))):
    post, x_map, mean, std = summarise(pr)
    assert post is not ... and mean is not ... and std is not ..., "fill in summarise() above first"
    results[name] = (x_map, mean, std)
    a.contourf(g, g, post.T, 20); a.plot(*x_map, "r+", ms=12); a.set_title(f"prior {name}\nMAP {np.round(x_map, 2)}, std {np.round(std, 2)}")
    a.set_xlabel("x1"); a.set_ylabel("x2"); a.set_xlim(-2, 2); a.set_ylim(-2, 2)
plt.tight_layout(); plt.show()

# check
(map_A, mean_A, std_A), (map_B, mean_B, std_B) = results.values()
assert np.allclose(map_A, [0.5, 0.5], atol=0.03) and np.allclose(map_B, [0.5, 0.5], atol=0.03), "both MAPs sit at (0.5, 0.5)"
assert np.all(std_B < std_A), "the smoothness prior should make each pixel LESS uncertain"
assert abs(std_A[0] - 0.71) < 0.03, "under prior A each pixel has std ~0.71: the unmeasured direction keeps its prior width"
print("exercise 3 OK ·  per-pixel std: A", np.round(std_A, 2), " B", np.round(std_B, 2))

## What you just did *is* the project

| here | at the school |
|---|---|
| `(x₁, x₂)` | a 128 × 128 image |
| `y = x₁ + x₂` | `y = M ⊙ fft2c(x)`, the undersampled k-space |
| prior A / prior B | a β-VAE decoder trained on knees |
| the MAP (red cross) | the MAP reconstruction (notebook 03) |
| the per-pixel std | the uncertainty map (notebook 04) |

One thing does **not** scale: the grid. Two pixels needed 801² ≈ 640 000 grid points; 128 latent
dimensions would need 801¹²⁸. That is why the real thing uses **MCMC** — a sampler that explores the
posterior instead of tabulating it. Next notebook.

## Done when

- all three checks print OK;
- you can explain the difference between the MAP and the posterior mean, and why prior B reduced the
  per-pixel uncertainty (it removed the direction the data never measured).